# India Recession Predictor — Full 3-Layer System (Jan 2000 – Feb 2026)

**This notebook replaces both `Train_Extended_Model_2000.ipynb` and the old
`Build_Full_System_2000(1).ipynb`.** Once this runs clean end-to-end, delete those two —
this is the single canonical Layer 1 pipeline going forward.

What it combines from each of its two predecessors:
- From `Build_Full_System_2000(1).ipynb`: merging macro data (IIP/CPI/GDP/repo rate/credit
  growth/unemployment/FinBERT sentiment, 2012-2026) with market data, momentum/acceleration
  features, and the GradientBoosting model with a precision-floor-tuned threshold (0.32,
  found via sweep during earlier development — kept as a constant here rather than
  re-sweeping every run).
- From the `Train_Extended_Model_2000.ipynb` fix: **real market data back to 2000**, not
  backfilled constants — USD/INR and Brent oil sourced from FRED (real coverage to
  1994/1987), Nifty spliced with BSE Sensex (a real, different index, not a synthetic
  proxy) for the 2000-2007 window before Nifty's own Yahoo history begins. Only India VIX
  remains genuinely unfixable (the instrument didn't exist before Nov 2007) and is flagged
  as a proxy rather than presented as real.

**Layer 1 — ML:** GradientBoosting on merged macro (2012-2026) + market (2000-2026, real) data
**Layer 2 — NLP:** FinBERT sentiment scoring (lightweight inline version — the full
  document-corpus pipeline with FAISS-ready output lives in `Layer2_FinBERT_Pipeline.ipynb`)
**Layer 3 — Agentic:** 5 rule-based risk agents, blended with ML (60/40) (lightweight inline
  version — the full FAISS-retrieval version lives in `Layer3_Agentic_FAISS.ipynb`)

Run top to bottom. Update `BACKEND_DIR` in Cell 1 if your path differs. Requires your
existing macro dataset (e.g. `15_master_ft_finbert.csv`) already in `backend/data/`.

In [1]:
# CELL 1 — Setup
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "yfinance", "pandas-datareader", "scikit-learn", "imbalanced-learn",
                "pandas", "numpy", "--quiet"])

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

BACKEND_DIR = Path(r"C:\final project\backend")
DATA_DIR    = BACKEND_DIR / "data"
MODEL_DIR   = BACKEND_DIR / "models"
POLICY_DIR  = BACKEND_DIR / "policy_docs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
POLICY_DIR.mkdir(parents=True, exist_ok=True)

print("Backend dir:", BACKEND_DIR)
print("Data dir   :", DATA_DIR)
print("Model dir  :", MODEL_DIR)

Backend dir: C:\final project\backend
Data dir   : C:\final project\backend\data
Model dir  : C:\final project\backend\models


## Layer 1a — Load your existing macro + FinBERT dataset (2012-2026)

In [2]:
# CELL 2 — Load your existing macro dataset
# This should already have IIP, CPI, GDP, repo rate, yield spread, credit
# growth, unemployment, and sent_mpc (FinBERT MPC sentiment) merged in,
# from your earlier notebooks (RM_Part_1 / RM_Scale_2).
#
# If this file lives somewhere else, update the path below.
MACRO_CANDIDATES = [
    DATA_DIR / "15_master_ft_finbert.csv",
    DATA_DIR / "14_master_scaled_features.csv",
    DATA_DIR / "13_master_ml_enhanced.csv",
    DATA_DIR / "12_master_ml_dataset.csv",
]

macro_path = next((p for p in MACRO_CANDIDATES if p.exists()), None)
if macro_path is None:
    raise FileNotFoundError(
        "No macro dataset found in backend/data/. Copy your "
        "15_master_ft_finbert.csv (or similar) there first."
    )

macro = pd.read_csv(macro_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(f"Loaded: {macro_path.name}")
print(f"Shape : {macro.shape}")
print(f"Range : {macro['date'].min().date()} to {macro['date'].max().date()}")
macro.columns.tolist()

Loaded: 15_master_ft_finbert.csv
Shape : (170, 29)
Range : 2012-01-01 to 2026-02-01


['date',
 'recession_label',
 'iip_growth_pct',
 'cpi_yoy_pct',
 'gdp_growth_pct',
 'repo_rate_pct',
 'yield_spread_bps',
 'credit_growth_yoy_pct',
 'unemployment_rate_pct',
 'sentiment_score',
 'iip_3m_rolling_avg',
 'cpi_momentum_3m',
 'repo_rate_change',
 'gdp_lag1q',
 'iip_lag1m',
 'yield_spread_lag1',
 'year',
 'month',
 'fiscal_year',
 'finbert_composite',
 'sent_mpc',
 'sent_budget',
 'sent_rbi_governor',
 'sent_econ_survey',
 'ft_composite',
 'ft_mpc',
 'ft_budget',
 'ft_rbi_governor',
 'ft_econ_survey']

## Layer 1b — Download market data, Jan 2000 - present (real data, not backfilled)

Builds the full monthly date scaffold first (so later merges actually have somewhere to
land — a naive `pd.concat` of partial-history series caps the whole dataset at whichever
series starts latest), then layers in FRED for INR/oil and Sensex for the pre-Nifty
equity-return gap. VIX remains a documented proxy pre-2008 — the one gap nothing can fix.

In [3]:
# CELL 3 — Market data 2000-2026, sourced from Yahoo + FRED + Sensex for real coverage
import yfinance as yf

START, END = "2000-01-01", "2026-03-01"

# Full monthly scaffold FIRST — later merges (FRED, Sensex) need real rows to land on.
full_dates = pd.date_range(start=START, end=END, freq="MS")
market = pd.DataFrame({"date": full_dates})

def add_yahoo_series(df, ticker, name):
    try:
        df_t = yf.download(ticker, start=START, end=END, interval="1mo", auto_adjust=True, progress=False)
        if len(df_t) == 0:
            print(f"  {name} -> no data returned")
            return df
        if isinstance(df_t.columns, pd.MultiIndex):
            df_t.columns = df_t.columns.get_level_values(0)
        s = df_t["Close"].resample("MS").last()
        s.index = pd.to_datetime(s.index)
        s_df = s.reset_index()
        s_df.columns = ["date", name]
        df = df.merge(s_df, on="date", how="left")
        print(f"  {name:<12} -> {s.notna().sum()} months (from {s.index.min().date()})")
        return df
    except Exception as e:
        print(f"  {name} -> failed: {str(e)[:60]}")
        return df

print("Downloading market data from Yahoo Finance (baseline)...")
market = add_yahoo_series(market, "USDINR=X", "usdinr")
market = add_yahoo_series(market, "^INDIAVIX", "india_vix")
market = add_yahoo_series(market, "BZ=F", "brent_oil")
market = add_yahoo_series(market, "^NSEI", "nifty50")

# --- Extend INR and oil with real FRED history (back to 1994 / 1987) --------
print("\nExtending INR and oil with FRED (real data)...")
try:
    import pandas_datareader.data as web
    fred_inr = web.DataReader("DEXINUS", "fred", "1994-01-01", END)
    fred_inr_m = fred_inr.resample("MS").last().reset_index()
    fred_inr_m.columns = ["date", "usdinr_fred"]
    market = market.merge(fred_inr_m, on="date", how="left")
    before = market["usdinr"].notna().sum()
    market["usdinr"] = market["usdinr"].fillna(market["usdinr_fred"])
    market = market.drop(columns=["usdinr_fred"])
    after = market["usdinr"].notna().sum()
    real_start = market.loc[market["usdinr"].notna(), "date"].min()
    print(f"  USD/INR (FRED DEXINUS): {after - before} additional real months filled, "
          f"now from {real_start.date() if pd.notna(real_start) else 'N/A'}")
except Exception as e:
    print(f"  FRED INR extension failed: {str(e)[:150]}")

try:
    import pandas_datareader.data as web
    for fred_code in ["DCOILBRENTEU", "DCOILWTICO"]:
        try:
            fred_oil = web.DataReader(fred_code, "fred", "1987-01-01", END)
            fred_oil_m = fred_oil.resample("MS").last().reset_index()
            fred_oil_m.columns = ["date", "brent_oil_fred"]
            market = market.merge(fred_oil_m, on="date", how="left")
            before = market["brent_oil"].notna().sum()
            market["brent_oil"] = market["brent_oil"].fillna(market["brent_oil_fred"])
            market = market.drop(columns=["brent_oil_fred"])
            after = market["brent_oil"].notna().sum()
            real_start = market.loc[market["brent_oil"].notna(), "date"].min()
            print(f"  Oil ({fred_code}): {after - before} additional real months filled, "
                  f"now from {real_start.date() if pd.notna(real_start) else 'N/A'}")
            break
        except Exception:
            continue
except Exception as e:
    print(f"  FRED oil extension failed: {str(e)[:150]}")

# --- Extend Nifty's pre-history gap with BSE Sensex (^BSESN) ----------------
# Real, different Indian equity index — used for the RETURN series (Sensex/Nifty
# levels differ ~3x in scale, so only returns are spliced, in CELL 5).
print("\nDownloading BSE Sensex (^BSESN) to cover Nifty's pre-history gap...")
try:
    sensex = yf.download("^BSESN", start=START, end=END, interval="1mo", auto_adjust=True, progress=False)
    if len(sensex) > 0:
        if isinstance(sensex.columns, pd.MultiIndex):
            sensex.columns = sensex.columns.get_level_values(0)
        sensex_s = sensex["Close"].resample("MS").last()
        sensex_s.index = pd.to_datetime(sensex_s.index)
        sensex_df = sensex_s.reset_index()
        sensex_df.columns = ["date", "sensex"]
        market = market.merge(sensex_df, on="date", how="left")
        nifty_start = market.loc[market["nifty50"].notna(), "date"].min()
        sensex_start = market.loc[market["sensex"].notna(), "date"].min()
        print(f"  Nifty real coverage from  {nifty_start.date() if pd.notna(nifty_start) else 'N/A'}")
        print(f"  Sensex real coverage from {sensex_start.date() if pd.notna(sensex_start) else 'N/A'} "
              f"— used as return-level substitute before Nifty's own start (flagged in CELL 5)")
    else:
        print("  Sensex download returned no data — Nifty stays limited to its own Yahoo history.")
except Exception as e:
    print(f"  Sensex download failed: {str(e)[:150]} — Nifty stays limited to its own Yahoo history.")

print(f"\nFinal market data shape: {market.shape}")
market.head()

  usdinr       -> 267 months (from 2003-12-01)
  india_vix    -> 216 months (from 2008-03-01)
  brent_oil    -> 191 months (from 2007-08-01)
  nifty50      -> 222 months (from 2007-09-01)

Extending INR and oil with FRED (real data)...
  USD/INR (FRED DEXINUS): 47 additional real months filled, now from 2000-01-01
  Oil (DCOILBRENTEU): 123 additional real months filled, now from 2000-01-01

  Nifty real coverage from  2007-09-01
  Sensex real coverage from 2000-01-01 — used as return-level substitute before Nifty's own start (flagged in CELL 5)

Final market data shape: (315, 6)


,date,usdinr,india_vix,brent_oil,nifty50,sensex
0,2000-01-01,43.65,NaN,27.08,NaN,5205.290039
1,2000-02-01,43.65,NaN,29.01,NaN,5447.470215
2,2000-03-01,43.65,NaN,23.98,NaN,5001.279785
3,2000-04-01,43.70,NaN,23.79,NaN,4657.549805
4,2000-05-01,44.65,NaN,29.64,NaN,4433.609863


## Layer 1c — Recession labels for all 302 months, 6 episodes

In [4]:
# CELL 4 — Six recession episodes (per PRD section 1.2)
RECESSION_PERIODS = [
    ("2001-09-01", "2002-03-01"),  # Dot-com + 9/11
    ("2008-10-01", "2009-03-01"),  # Global Financial Crisis
    ("2013-06-01", "2013-09-01"),  # Taper tantrum
    ("2016-11-01", "2017-02-01"),  # Demonetization
    ("2019-07-01", "2019-11-01"),  # Pre-COVID slowdown
    ("2020-03-01", "2020-10-01"),  # COVID-19
]

date_range = pd.date_range(start="2000-01-01", end="2026-02-01", freq="MS")
labels = pd.DataFrame({"date": date_range, "recession_label": 0})
for start, end in RECESSION_PERIODS:
    mask = (labels["date"] >= start) & (labels["date"] <= end)
    labels.loc[mask, "recession_label"] = 1

print(f"Total months: {len(labels)} | Recession months: {labels['recession_label'].sum()}")

Total months: 314 | Recession months: 34


## Layer 1d — Merge everything into one 302+-month dataset

Market features now use real data (INR, oil, and equity returns spliced Nifty+Sensex).
Macro indicators (IIP/CPI/GDP/repo rate etc.) genuinely don't exist before ~2011-2012 —
those are still flat-filled from the earliest real reading and flagged with `_is_proxy`
columns, same honest disclosure pattern as your TRD's gap-handling section. VIX is flagged
the same way for its pre-2008 gap.

In [5]:
# CELL 5 — Merge labels + market (real, full history) + macro (2012-2026)
full = labels.merge(market, on="date", how="left")
full = full.merge(macro.drop(columns=["recession_label"], errors="ignore"),
                   on="date", how="left")

# --- INR: real from 2000-01 via FRED, no proxy needed -----------------------
if "usdinr" in full.columns:
    full["inr_level"]           = full["usdinr"]
    full["inr_depreciation_1m"] = full["usdinr"].pct_change(1, fill_method=None) * 100
    full["inr_depreciation_3m"] = full["usdinr"].pct_change(3, fill_method=None) * 100

# --- Oil: real from 2000-01 via FRED, no proxy needed ------------------------
if "brent_oil" in full.columns:
    full["oil_price_usd"] = full["brent_oil"]
    full["oil_change_3m"] = full["brent_oil"].pct_change(3, fill_method=None) * 100

# --- Equity returns: splice real Nifty (from ~2007-09) with real Sensex ------
# (from 2000-01) for the pre-Nifty window, spliced at the RETURN level (not raw
# price level, since the two indices differ ~3x in scale).
if "nifty50" in full.columns and "sensex" in full.columns:
    nifty_ret_1m  = full["nifty50"].pct_change(1, fill_method=None)
    sensex_ret_1m = full["sensex"].pct_change(1, fill_method=None)

    nifty_start = full.loc[full["nifty50"].notna(), "date"].min()
    equity_is_proxy = full["date"] < nifty_start
    combined_ret_1m = nifty_ret_1m.where(~equity_is_proxy, sensex_ret_1m)
    full["equity_return_is_proxy"] = equity_is_proxy

    synthetic_index = 100 * (1 + combined_ret_1m.fillna(0)).cumprod()
    full["nifty_return_1m"] = combined_ret_1m * 100
    full["nifty_return_3m"] = synthetic_index.pct_change(3, fill_method=None) * 100
    full["nifty_3m_vol"]    = combined_ret_1m.rolling(3).std() * 100

    n_proxy = equity_is_proxy.sum()
    print(f"Equity return features added ({n_proxy} months before {nifty_start.date()} "
          f"use Sensex-derived returns, flagged in equity_return_is_proxy)")
elif "nifty50" in full.columns:
    combined_ret_1m = full["nifty50"].pct_change(1, fill_method=None)
    full["nifty_return_1m"] = combined_ret_1m * 100
    full["nifty_return_3m"] = full["nifty50"].pct_change(3, fill_method=None) * 100
    full["nifty_3m_vol"]    = combined_ret_1m.rolling(3).std() * 100

# --- VIX: still no real fix (didn't exist before Nov 2007), but now built on
# the real Sensex-extended return series rather than a raw-Nifty-only proxy.
if "india_vix" in full.columns:
    nifty_realized_vol = combined_ret_1m.rolling(3).std() * 100 * np.sqrt(12)
    india_vix_start = full.loc[full["india_vix"].notna(), "date"].min()
    vix_is_proxy = full["date"] < india_vix_start

    full["vix_level"] = full["india_vix"].where(~vix_is_proxy, nifty_realized_vol)
    full["vix_level_is_proxy"] = vix_is_proxy
    full["vix_change"] = full["vix_level"].diff()
    full["vix_3m_avg"]  = full["vix_level"].rolling(3).mean()

    n_vix_proxy = vix_is_proxy.sum()
    print(f"VIX features added ({n_vix_proxy} months before {india_vix_start.date()} "
          f"use Sensex/Nifty-realized-volatility proxy — the one gap with no real-data "
          f"fix, since India VIX did not exist before Nov 2007)")

# GAP HANDLING (per TRD section 3.2):
# sent_mpc doesn't exist before Oct 2016 (MPC wasn't established yet) -> fill 0 (neutral)
if "sent_mpc" in full.columns:
    full["sent_mpc_available"] = full["sent_mpc"].notna()
    full["sent_mpc"] = full["sent_mpc"].fillna(0.0)

# Macro indicators (IIP/CPI/GDP/repo/yield/credit) genuinely don't exist before
# ~2011-2012 in your macro dataset — flat-filled from the earliest real reading
# and flagged, same disclosure pattern as the market features above.
macro_cols = [c for c in macro.columns if c not in ("date", "recession_label")]
for c in macro_cols:
    if c in full.columns:
        full[f"{c}_is_proxy"] = full[c].isna()
full[macro_cols] = full[macro_cols].bfill().ffill()

n_before = int(full.isna().sum().sum())
full = full.ffill().bfill()

out_path = DATA_DIR / "extended_2000_2026.csv"
full.to_csv(out_path, index=False)
print(f"\nResidual ffill/bfill patched {n_before} remaining NaN cells "
      f"(rolling-window edges etc. — should be small relative to {full.shape[0]}x{full.shape[1]})")
print(f"Final merged shape: {full.shape}")
print(f"Saved to: {out_path}")

Equity return features added (92 months before 2007-09-01 use Sensex-derived returns, flagged in equity_return_is_proxy)
VIX features added (98 months before 2008-03-01 use Sensex/Nifty-realized-volatility proxy — the one gap with no real-data fix, since India VIX did not exist before Nov 2007)

Residual ffill/bfill patched 232 remaining NaN cells (rolling-window edges etc. — should be small relative to 314x75)
Final merged shape: (314, 75)
Saved to: C:\final project\backend\data\extended_2000_2026.csv


In [6]:
# CELL 5b — Confirm the fix: pre-2008 values should vary now, not repeat a constant
for col in ["inr_level", "oil_price_usd", "nifty_return_1m", "vix_level"]:
    if col not in full.columns:
        continue
    pre_2008 = full[full["date"] < "2008-01-01"][col]
    print(f"{col:<18} pre-2008 unique values: {pre_2008.nunique():>4} / {len(pre_2008)} months")
print()
print("inr_level / oil_price_usd / nifty_return_1m should show many unique values (real data).")
print("vix_level will show low variation before ~2007 — that's the honest, correctly-flagged")
print("remaining gap (vix_level_is_proxy), not a bug.")

inr_level          pre-2008 unique values:   94 / 96 months
oil_price_usd      pre-2008 unique values:   96 / 96 months
nifty_return_1m    pre-2008 unique values:   94 / 96 months
vix_level          pre-2008 unique values:   90 / 96 months

inr_level / oil_price_usd / nifty_return_1m should show many unique values (real data).
vix_level will show low variation before ~2007 — that's the honest, correctly-flagged
remaining gap (vix_level_is_proxy), not a bug.


## Layer 1e — Feature engineering + train GradientBoosting on all months

Momentum/acceleration features first, then the final model. GradientBoosting with a
precision-floor-0.32-tuned threshold was chosen after comparing RandomForest,
GradientBoosting, and LogisticRegression plus a majority-vote ensemble during earlier
development (see git history / earlier notebook versions for that sweep) — GradientBoosting
gave the best recall/false-alarm tradeoff at this floor, so it's used directly here rather
than re-running the full comparison sweep on every execution.

In [7]:
# CELL 6 — Add momentum/acceleration features (helps catch building stress
# before a recession peaks, not just during the peak itself)
for col in ["vix_level", "inr_level", "oil_price_usd"]:
    if col in full.columns:
        full[f"{col}_chg_6m"] = full[col].pct_change(6, fill_method=None) * 100
        full[f"{col}_accel"]  = full[col].diff().diff()  # 2nd derivative = acceleration

# Nifty renamed to nifty50 historically but now spliced — use the synthetic index
if "nifty50" in full.columns:
    full["nifty50_chg_6m"] = full["nifty50"].pct_change(6, fill_method=None) * 100
    full["nifty50_accel"]  = full["nifty50"].diff().diff()

for col in ["repo_rate_pct", "credit_growth_yoy_pct", "iip_growth_pct"]:
    if col in full.columns:
        full[f"{col}_chg_6m"] = full[col].diff(6)

full = full.replace([np.inf, -np.inf], np.nan)
full = full.ffill().bfill()
print(f"Feature count now: {full.shape[1]}")

Feature count now: 86


In [8]:
# CELL 7 — FINAL: GradientBoosting, regularized against overfitting (was Train AUC 1.000,
# Test AUC 0.734 — a 0.266 gap, confirming the model was memorizing train rather than
# generalizing). Fixes: row subsampling, feature subsampling per split, early stopping.
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import precision_recall_curve, roc_auc_score

EXCLUDE = {"date", "recession_label"} | {c for c in full.columns if c.endswith("_is_proxy")} | {"sent_mpc_available"}
FEATURES = [c for c in full.columns if c not in EXCLUDE and full[c].dtype != object]
TARGET = "recession_label"

df_clean = full[["date"] + FEATURES + [TARGET]].dropna().reset_index(drop=True)
X = df_clean[FEATURES].values
y = df_clean[TARGET].values
dates = pd.to_datetime(df_clean["date"]).values

bad_cols = [c for c in FEATURES if not np.isfinite(df_clean[c]).all()]
if bad_cols:
    raise ValueError(f"Non-finite values remain in: {bad_cols} — inspect before scaling.")

SPLIT = int(len(y) * 0.70)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, y_train = X_scaled[:SPLIT], y[:SPLIT]
X_test,  y_test  = X_scaled[SPLIT:], y[SPLIT:]

print(f"Dataset: {len(df_clean)} months, {len(FEATURES)} features")
print(f"Train  : {SPLIT} months ({y_train.sum()} recession) | {dates[0]} to {dates[SPLIT-1]}")
print(f"Test   : {len(y_test)} months ({y_test.sum()} recession) | {dates[SPLIT]} to {dates[-1]}")

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42,
    subsample=0.7,             # each tree trained on a random 70% of rows — can't memorize all of them
    max_features=0.6,          # each split only considers 60% of the 52 features — reduces reliance on 1-2 dominant features
    min_samples_leaf=5,        # a leaf needs at least 5 samples — blocks tiny memorized pockets
    validation_fraction=0.15,  # holds out 15% of the (already-chronological) train data
    n_iter_no_change=15,       # stops adding trees after 15 rounds without validation improvement
    tol=1e-4,
)
model.fit(X_res, y_res)
print(f"\nStopped at {model.n_estimators_} trees (of {model.n_estimators} max) via early stopping")

probs = model.predict_proba(X_scaled)[:, 1]
probs_train = probs[:SPLIT]
probs_test = probs[SPLIT:]
train_rec_idx = np.where(y_train == 1)[0]
test_rec_idx  = np.where(y_test == 1)[0]

train_auc = roc_auc_score(y_train, probs_train)
test_auc  = roc_auc_score(y_test, probs_test)
print(f"Train AUC: {train_auc:.3f} | Test AUC: {test_auc:.3f} | Gap: {train_auc - test_auc:.3f}")

prec, rec, thresh = precision_recall_curve(y_train, probs_train)
valid = prec[:-1] >= 0.32
BEST_THRESHOLD = thresh[np.where(valid)[0][np.argmax(rec[:-1][valid])]] if valid.any() else 0.50

train_caught = (probs_train[train_rec_idx] >= BEST_THRESHOLD).sum()
test_caught  = (probs_test[test_rec_idx] >= BEST_THRESHOLD).sum()
train_fa = ((y_train == 0) & (probs_train >= BEST_THRESHOLD)).sum() / (y_train == 0).sum()
test_fa  = ((y_test == 0) & (probs_test >= BEST_THRESHOLD)).sum() / (y_test == 0).sum()

print(f"\nModel: GradientBoosting (regularized) | Threshold: {BEST_THRESHOLD:.3f}")
print(f"Train: {train_caught}/{len(train_rec_idx)} caught, {train_fa:.1%} FA")
print(f"Test : {test_caught}/{len(test_rec_idx)} caught, {test_fa:.1%} FA")

print("\nTop features:")
for i in np.argsort(model.feature_importances_)[::-1][:10]:
    bar = chr(9608) * int(model.feature_importances_[i] * 200)
    print(f"  {FEATURES[i]:<28} {model.feature_importances_[i]:.3f} {bar}")

Dataset: 314 months, 52 features
Train  : 219 months (21 recession) | 2000-01-01T00:00:00.000000000 to 2018-03-01T00:00:00.000000000
Test   : 95 months (13 recession) | 2018-04-01T00:00:00.000000000 to 2026-02-01T00:00:00.000000000

Stopped at 122 trees (of 200 max) via early stopping
Train AUC: 1.000 | Test AUC: 0.692 | Gap: 0.307

Model: GradientBoosting (regularized) | Threshold: 0.025
Train: 21/21 caught, 22.2% FA
Test : 10/13 caught, 64.6% FA

Top features:
  usdinr                       0.202 ████████████████████████████████████████
  nifty50_chg_6m               0.200 ████████████████████████████████████████
  inr_level                    0.191 ██████████████████████████████████████
  inr_depreciation_3m          0.067 █████████████
  inr_level_chg_6m             0.065 ████████████
  gdp_growth_pct               0.058 ███████████
  iip_growth_pct_chg_6m        0.036 ███████
  repo_rate_pct_chg_6m         0.036 ███████
  vix_3m_avg                   0.021 ████
  cpi_momentum_3m  

In [9]:
# Quick check — does GradientBoosting's train-vs-test gap look like overfitting?
from sklearn.metrics import roc_auc_score
train_auc = roc_auc_score(y_train, probs_train)
test_auc  = roc_auc_score(y_test, probs_test)
print(f"Train AUC: {train_auc:.3f}")
print(f"Test AUC : {test_auc:.3f}")
print(f"Gap      : {train_auc - test_auc:.3f}  (>0.15-0.20 gap is a real overfitting signal)")

Train AUC: 1.000
Test AUC : 0.692
Gap      : 0.307  (>0.15-0.20 gap is a real overfitting signal)


In [10]:
# CELL 8 — Save Layer 1 (GradientBoosting, trained on real 2000-2026 market data)
pickle.dump(model,    open(MODEL_DIR / "rf_extended_2000.pkl", "wb"))
pickle.dump(scaler,   open(MODEL_DIR / "scaler_extended_2000.pkl", "wb"))
pickle.dump(FEATURES, open(MODEL_DIR / "features_extended_2000.pkl", "wb"))
pickle.dump(BEST_THRESHOLD, open(MODEL_DIR / "threshold_extended_2000.pkl", "wb"))
print(f"Layer 1 saved (GradientBoosting). Threshold: {BEST_THRESHOLD:.3f}")
print(f"Test performance: {test_caught}/{len(test_rec_idx)} recall ({test_caught/max(len(test_rec_idx),1):.1%}), "
      f"{test_fa:.1%} false alarm rate")

Layer 1 saved (GradientBoosting). Threshold: 0.025
Test performance: 10/13 recall (76.9%), 64.6% false alarm rate


## Layer 2 — NLP (FinBERT sentiment scoring, lightweight inline version)

For the full document-corpus pipeline (RBI/Budget/Economic Survey scraping, chunking,
weighted monthly composite), use `Layer2_FinBERT_Pipeline.ipynb` instead — this cell just
defines the scoring function for quick/ad-hoc use.

In [11]:
# CELL 9 — FinBERT scoring function (lightweight Layer 2 reference)
def load_finbert():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    mdl = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
    mdl.eval()
    return tok, mdl

def score_text(text, tok, mdl):
    import torch
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = mdl(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    # FinBERT labels: 0=positive, 1=negative, 2=neutral
    score = float(probs[0] - probs[1])  # positive - negative, range -1 to +1
    return score

# Uncomment to test on a sample sentence (downloads FinBERT the first time):
# tok, mdl = load_finbert()
# print(score_text("The economy is showing signs of stress and slowing growth.", tok, mdl))

print("Layer 2 (NLP) reference ready. For the full pipeline, use Layer2_FinBERT_Pipeline.ipynb.")

Layer 2 (NLP) reference ready. For the full pipeline, use Layer2_FinBERT_Pipeline.ipynb.


## Layer 3 — Agentic system (5 rule-based agents, lightweight inline version)

For the full FAISS-retrieval agentic system (6 specialist agents including Market Stress
and Autoencoder Anomaly, real document evidence retrieval), use `Layer3_Agentic_FAISS.ipynb`
and `Layer4_DeepLearning_GenAI.ipynb` instead — this cell is a quick transparent reference.

In [12]:
# CELL 10 — 5 rule-based agents, blended 60% ML / 40% Agents
def agent_monetary_policy(row):
    repo = row.get("repo_rate_pct", 0) or 0
    sent = row.get("sent_mpc", 0) or 0
    return min(1.0, max(0.0, (repo - 4) / 6 - sent * 0.3))

def agent_yield_curve(row):
    spread = row.get("yield_spread_bps", row.get("yield_spread_lag1", 0)) or 0
    return min(1.0, max(0.0, -spread / 200))

def agent_credit_conditions(row):
    credit = row.get("credit_growth_yoy_pct", 10) or 10
    return min(1.0, max(0.0, (10 - credit) / 10))

def agent_market_stress(row):
    vix = row.get("vix_level", 15) or 15
    inr_dep = row.get("inr_depreciation_3m", 0) or 0
    return min(1.0, max(0.0, (vix - 15) / 25 + inr_dep / 20))

def agent_industrial_activity(row):
    iip = row.get("iip_growth_pct", row.get("iip_3m_rolling_avg", 2)) or 2
    return min(1.0, max(0.0, (2 - iip) / 8))

AGENTS = [agent_monetary_policy, agent_yield_curve, agent_credit_conditions,
          agent_market_stress, agent_industrial_activity]

def run_agents(row):
    scores = [a(row) for a in AGENTS]
    return float(np.mean(scores)), scores

agent_scores = full.apply(lambda r: run_agents(r)[0], axis=1)

df_clean["agent_score"] = agent_scores.iloc[df_clean.index].values
df_clean["ml_probability"] = probs
df_clean["final_probability"] = df_clean["ml_probability"] * 0.60 + df_clean["agent_score"] * 0.40
df_clean["status"] = np.select(
    [df_clean["final_probability"] >= BEST_THRESHOLD, df_clean["final_probability"] >= BEST_THRESHOLD * 0.5],
    ["alert", "watch"], default="normal"
)

print("Layer 3 (Agentic) blend complete.")
df_clean[["date", "ml_probability", "agent_score", "final_probability", "status"]].tail(10)

Layer 3 (Agentic) blend complete.


,date,ml_probability,agent_score,final_probability,status
304,2025-05-01,0.073377,0.069593,0.071863,alert
305,2025-06-01,0.037363,0.079593,0.054255,alert
306,2025-07-01,0.016616,0.052613,0.031015,alert
307,2025-08-01,0.005685,0.055431,0.025583,alert
308,2025-09-01,0.004810,0.057008,0.025689,alert
309,2025-10-01,0.021552,0.090426,0.049102,alert
310,2025-11-01,0.026373,0.090426,0.051994,alert
311,2025-12-01,0.059607,0.090426,0.071935,alert
312,2026-01-01,0.211002,0.105519,0.168809,alert
313,2026-02-01,0.168982,0.090314,0.137515,alert


In [13]:
# CELL 11 — Save final blended predictions for the backend to serve
final_out = df_clean[["date", "recession_label", "ml_probability", "agent_score",
                       "final_probability", "status"]].copy()
final_out["date"] = pd.to_datetime(final_out["date"]).dt.strftime("%Y-%m-%d")

out_path = DATA_DIR / "final_predictions_2000_2026.csv"
final_out.to_csv(out_path, index=False)

caught = ((final_out["final_probability"] >= BEST_THRESHOLD) & (final_out["recession_label"] == 1)).sum()
total_recession = final_out["recession_label"].sum()

print("All 3 layers complete:")
print(f"  Layer 1 (ML)      -> {MODEL_DIR / 'rf_extended_2000.pkl'}")
print(f"  Layer 2 (NLP)     -> score_text() function defined above")
print(f"  Layer 3 (Agentic) -> {out_path}")
print()
print(f"Alert threshold used: {BEST_THRESHOLD:.3f}")
print(f"Recession months caught (final, >={BEST_THRESHOLD:.2f}): {caught}/{total_recession}")
print()
print("Next: re-run Layer2_FinBERT_Pipeline.ipynb -> Layer3_Agentic_FAISS.ipynb ->")
print("Layer4_DeepLearning_GenAI.ipynb, in that order, then restart your Flask backend.")

All 3 layers complete:
  Layer 1 (ML)      -> C:\final project\backend\models\rf_extended_2000.pkl
  Layer 2 (NLP)     -> score_text() function defined above
  Layer 3 (Agentic) -> C:\final project\backend\data\final_predictions_2000_2026.csv

Alert threshold used: 0.025
Recession months caught (final, >=0.02): 34/34

Next: re-run Layer2_FinBERT_Pipeline.ipynb -> Layer3_Agentic_FAISS.ipynb ->
Layer4_DeepLearning_GenAI.ipynb, in that order, then restart your Flask backend.


In [14]:
# CELL 12 — Diagnose: which episodes is the model actually catching?
episode_names = {
    1: ("2001-09-01", "2002-03-01", "Dot-com + 9/11"),
    2: ("2008-10-01", "2009-03-01", "Global Financial Crisis"),
    3: ("2013-06-01", "2013-09-01", "Taper tantrum"),
    4: ("2016-11-01", "2017-02-01", "Demonetization"),
    5: ("2019-07-01", "2019-11-01", "Pre-COVID slowdown"),
    6: ("2020-03-01", "2020-10-01", "COVID-19"),
}

df_clean["date"] = pd.to_datetime(df_clean["date"])

for ep, (start, end, name) in episode_names.items():
    mask = (df_clean["date"] >= start) & (df_clean["date"] <= end)
    sub = df_clean[mask]
    caught = (sub["final_probability"] >= 0.60).sum()
    total = len(sub)
    avg_p = sub["final_probability"].mean()
    print(f"Episode {ep} ({name}): {caught}/{total} caught | avg P = {avg_p:.3f}")

Episode 1 (Dot-com + 9/11): 7/7 caught | avg P = 0.708
Episode 2 (Global Financial Crisis): 6/6 caught | avg P = 0.759
Episode 3 (Taper tantrum): 4/4 caught | avg P = 0.698
Episode 4 (Demonetization): 3/4 caught | avg P = 0.610
Episode 5 (Pre-COVID slowdown): 0/5 caught | avg P = 0.216
Episode 6 (COVID-19): 3/8 caught | avg P = 0.443
